# Video RAG — Phase 1: Notebook 02
## Timestamp-Preserving Transcript Chunking

In this notebook, we transform the raw sequence of speech segments into **semantically coherent, timestamp-bounded chunks**.

### Key Rules:
- Chunks must strictly preserve start and end timestamps (`start_time`, `end_time`).
- Chunk size is configurable (target 300–500 tokens/words).
- Overlap is configurable to maintain context across boundaries.


In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.transcription import Transcriber, format_timestamp
from src.chunking import TranscriptChunker


### Step 1: Load Timestamped Transcript JSON


In [ ]:
transcript_json_path = "../data/transcripts/sample_transcript.json"

segments = Transcriber.load_transcript_json(transcript_json_path)
print(f"Loaded {len(segments)} segments from {transcript_json_path}")


### Step 2: Configure and Execute Sliding-Window Chunker


In [ ]:
# Initialize chunker with target token size of ~350 and overlap of ~50
chunker = TranscriptChunker(
    target_chunk_size=350,
    chunk_overlap=50,
    min_chunk_size=40
)

chunks = chunker.chunk_segments(segments, source_video="sample.mp4")
print(f"Generated {len(chunks)} timestamped chunks.")


### Step 3: Inspect Generated Chunks and Temporal Boundaries


In [ ]:
chunk_records = [
    {
        "Chunk ID": c.chunk_id,
        "Time Range": c.formatted_time_range(),
        "Start (s)": c.start_time,
        "End (s)": c.end_time,
        "Duration (s)": round(c.duration, 2),
        "Approx Tokens": c.token_count_approx,
        "Snippet": c.text[:90] + "..."
    }
    for c in chunks
]
df_chunks = pd.DataFrame(chunk_records)
df_chunks


### Step 4: Persist Chunks to Disk


In [ ]:
chunks_output_path = "../data/transcripts/sample_chunks.json"
chunker.save_chunks(chunks, chunks_output_path)
print(f"Saved chunks to {chunks_output_path}")
